# Lab 1: Databricks Fundamentals
**Objective:** Establish DEV workflow, ingest Video Game Sales data, apply Spark transformations, and load external API data into Unity Catalog.

In [0]:
spark

In [0]:
games_df = spark.table("dbr_dev.ayyuborujzade.video_game_sales")

In [0]:
display(games_df.limit(10))

games_df.printSchema()
print(f"Rows: {games_df.count()}")
print(f"Columns: {len(games_df.columns)}")

In [0]:
selected_games = games_df.select(
    "Name",
    "Platform",
    "Genre",
    "Publisher",
    "Global_Sales"
)

display(selected_games)

In [0]:
top_games = games_df.filter(
    games_df.Global_Sales > 10
)

display(top_games)

In [0]:
from pyspark.sql.functions import sum

genre_summary = (
    games_df
    .groupBy("Genre")
    .agg(
        sum("Global_Sales").alias("Sum_Global_Sales")
    )
)

display(genre_summary)

In [0]:
genre_summary.withColumnRenamed("Sum_Global_Sales", "total_global_sales").write \
    .mode("overwrite") \
    .saveAsTable(
        "dbr_dev.ayyuborujzade.genre_sales_summary"
    )

**Dashboards**

- Games per genre

In [0]:
genre_sales_df = (
    games_df
    .groupBy("Genre")
    .agg(
        {"Global_Sales": "sum"}
    )
    .withColumnRenamed(
        "sum(Global_Sales)",
        "Total_Sales"
    )
)

display(genre_sales_df)

Databricks visualization. Run in Databricks to view.

- Sales by platform

In [0]:
platform_sales_df = (
    games_df
    .groupBy("Platform")
    .agg(
        {"Global_Sales": "sum"}
    )
    .withColumnRenamed(
        "sum(Global_Sales)",
        "Total_Sales"
    )
)

display(platform_sales_df)

Databricks visualization. Run in Databricks to view.

- Top publishers

In [0]:
publisher_sales_df = (
    games_df
    .groupBy("Publisher")
    .agg(
        {"Global_Sales": "sum"}
    )
    .withColumnRenamed(
        "sum(Global_Sales)",
        "Total_Sales"
    )
    .orderBy(
        "Total_Sales",
        ascending=False
    )
)

display(publisher_sales_df.limit(10))

Databricks visualization. Run in Databricks to view.

In [0]:
import requests

response = requests.get("https://jsonplaceholder.typicode.com/users")

users = response.json()

In [0]:
import pandas as pd

api_df = spark.createDataFrame(pd.json_normalize(users))

display(api_df)

## External API Data Ingestion

In this section, data is retrieved from a public API and converted into a Spark DataFrame.

In [0]:
import requests

In [0]:
api_url = "https://pokeapi.co/api/v2/pokemon?limit=50"

response = requests.get(api_url)

response.status_code

In [0]:
pokemon_json = response.json()

pokemon_json.keys()

In [0]:
pokemon_records = pokemon_json["results"]

pokemon_records[:5]

In [0]:
pokemon_df = spark.createDataFrame(pokemon_records)

display(pokemon_df)

In [0]:
from pyspark.sql.functions import upper

pokemon_clean_df = pokemon_df.withColumn(
    "pokemon_name_upper",
    upper(pokemon_df.name)
)

display(pokemon_clean_df)

In [0]:
pokemon_clean_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(
        "dbr_dev.ayyuborujzade.pokemon_api_data"
    )